In [105]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder


In [106]:
df = pd.read_csv("../Data Cleaning/cars_clean.csv")
df.head()

,id,make,model,year,km,price_egp,transmission,color,city,seller_type,fuel_type
0,7284417,Toyota,Yaris,2009,170000.0,540000,Automatic,Dark blue,Heliopolis,dealer,Petrol
1,7296785,Toyota,Camry,2013,113000.0,1000000,Automatic,White,Nasr city,private,Petrol
2,7293331,Toyota,Corolla,2024,95000.0,1380000,Automatic,Black,Mohandessin,private,Petrol
3,7288941,Toyota,Corolla,2008,210000.0,585000,Automatic,Black,Heliopolis,dealer,Petrol
4,7295928,Toyota,Corolla,2022,126000.0,980000,Automatic,Dark red,Maadi,dealer,Petrol


In [107]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3265 entries, 0 to 3264
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            3265 non-null   int64  
 1   make          3265 non-null   str    
 2   model         3265 non-null   str    
 3   year          3265 non-null   int64  
 4   km            3081 non-null   float64
 5   price_egp     3265 non-null   int64  
 6   transmission  3265 non-null   str    
 7   color         3207 non-null   str    
 8   city          3265 non-null   str    
 9   seller_type   3265 non-null   str    
 10  fuel_type     3265 non-null   str    
dtypes: float64(1), int64(3), str(7)
memory usage: 432.7 KB


## Fill_Reset_nulls

In [108]:
df['km'].isnull().sum()


np.int64(184)

In [109]:
# df['km']=df['km'].fillna(df['km'].median())

In [110]:
df['color'].isnull().sum()

np.int64(58)

In [111]:
# df['color']=df['color'].fillna(df['color'].mode()[0])

## Separate Features & Target

In [112]:
cat_cols = df.select_dtypes(include=['object']).columns

C:\Users\ot\AppData\Local\Temp\ipykernel_31904\336809641.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [113]:
df[cat_cols].nunique().sort_values(ascending=False)

city            169
model           121
color            25
fuel_type         5
make              4
transmission      2
seller_type       2
dtype: int64

In [115]:
X = df.drop(columns=['price_egp', 'id'])
y = df['price_egp']

In [116]:
# Hyundai: 1200 سيارة.

# Kia: 900 سيارة.

# Nissan: 600 سيارة.

# Other: 300 سيارة.

In [117]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 ,stratify=X['make']

)

In [118]:
color_mode = X_train['color'].mode()[0]

X_train['color'] = X_train['color'].fillna(color_mode)
X_test['color'] = X_test['color'].fillna(color_mode)

In [119]:
X_train['km'] = X_train['km'].fillna(X_train['km'].median())
X_test['km'] = X_test['km'].fillna(X_train['km'].median())
print(X_train['km'].isna().sum())
print(X_test['km'].isna().sum())

0
0


## Encoding

**OHE**

In [120]:
ohe_cols = ['make', 'transmission', 'seller_type', 'fuel_type', 'color']
freq_cols = ['model', 'city']

In [121]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_ohe = ohe.fit_transform(X_train[ohe_cols])
X_test_ohe = ohe.transform(X_test[ohe_cols])

ohe_names = ohe.get_feature_names_out(ohe_cols)

X_train_ohe = pd.DataFrame(
    X_train_ohe,
    columns=ohe_names,
    index=X_train.index
)

X_test_ohe = pd.DataFrame(
    X_test_ohe,
    columns=ohe_names,
    index=X_test.index
)

In [122]:
X_train = X_train.drop(columns=ohe_cols)
X_test = X_test.drop(columns=ohe_cols)

X_train = pd.concat([X_train, X_train_ohe], axis=1)
X_test = pd.concat([X_test, X_test_ohe], axis=1)

In [123]:
X_train

,model,year,km,city,make_Hyundai,make_Kia,make_Nissan,make_Toyota,transmission_Automatic,transmission_Manual,...,color_Mocha,color_Olive,color_Orange,color_Other,color_Petroleum,color_Purple,color_Red,color_Silver,color_White,color_Yellow
132,Corolla,2020,130000.0,Tagamo3 - New Cairo,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
954,Juke,2025,51000.0,Banha,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
52,Corolla,2018,100000.0,Smouha,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
998,Sunny,2018,169999.0,Madinaty,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
267,Avensis,2007,373000.0,Gharbia,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842,Qashqai,2019,162000.0,Warraq,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2324,Elantra Cn7,2021,95000.0,Cairo,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2596,Tucson Turbo,2017,80000.0,Mokattam,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2333,Ix 35,2014,280000.0,Cairo,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Frequency**

In [124]:
freq_cols = ['model', 'city']
for col in freq_cols:
    freq = X_train[col].value_counts(normalize=True)

    X_train[col] = X_train[col].map(freq)
    X_test[col] = X_test[col].map(freq).fillna(0)

In [125]:
X_train

,model,year,km,city,make_Hyundai,make_Kia,make_Nissan,make_Toyota,transmission_Automatic,transmission_Manual,...,color_Mocha,color_Olive,color_Orange,color_Other,color_Petroleum,color_Purple,color_Red,color_Silver,color_White,color_Yellow
132,0.073890,2020,130000.0,0.067764,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
954,0.009954,2025,51000.0,0.008806,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
52,0.073890,2018,100000.0,0.010337,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
998,0.108729,2018,169999.0,0.006508,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
267,0.000383,2007,373000.0,0.003446,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842,0.048622,2019,162000.0,0.001531,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2324,0.014548,2021,95000.0,0.140505,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2596,0.006508,2017,80000.0,0.008040,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2333,0.006126,2014,280000.0,0.140505,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [126]:
print(X_train.head())
print(X_train.dtypes)

        model  year        km      city  make_Hyundai  make_Kia  make_Nissan  \
132  0.073890  2020  130000.0  0.067764           0.0       0.0          0.0   
954  0.009954  2025   51000.0  0.008806           0.0       0.0          1.0   
52   0.073890  2018  100000.0  0.010337           0.0       0.0          0.0   
998  0.108729  2018  169999.0  0.006508           0.0       0.0          1.0   
267  0.000383  2007  373000.0  0.003446           0.0       0.0          0.0   

     make_Toyota  transmission_Automatic  transmission_Manual  ...  \
132          1.0                     1.0                  0.0  ...   
954          0.0                     1.0                  0.0  ...   
52           1.0                     1.0                  0.0  ...   
998          0.0                     1.0                  0.0  ...   
267          1.0                     0.0                  1.0  ...   

     color_Mocha  color_Olive  color_Orange  color_Other  color_Petroleum  \
132          0.0     

## Scaling

In [127]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train[['km','year']])
X_test_scaled = scaler.transform(X_test[['km','year']])

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=['km','year'],
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=['km','year'],
    index=X_test.index
)

In [128]:
X_train['km']=X_train_scaled['km']
X_test['km']=X_test_scaled['km']
X_train['year']=X_train_scaled['year']
X_test['year']=X_test_scaled['year']

In [129]:
X_train

,model,year,km,city,make_Hyundai,make_Kia,make_Nissan,make_Toyota,transmission_Automatic,transmission_Manual,...,color_Mocha,color_Olive,color_Orange,color_Other,color_Petroleum,color_Purple,color_Red,color_Silver,color_White,color_Yellow
132,0.073890,0.572442,-0.117864,0.067764,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
954,0.009954,1.210117,-0.869240,0.008806,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
52,0.073890,0.317373,-0.403197,0.010337,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
998,0.108729,0.317373,0.262570,0.006508,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
267,0.000383,-1.085512,2.193330,0.003446,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842,0.048622,0.444907,0.186491,0.001531,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2324,0.014548,0.699977,-0.450752,0.140505,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2596,0.006508,0.189838,-0.593419,0.008040,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2333,0.006126,-0.192767,1.308799,0.140505,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [130]:
df

,id,make,model,year,km,price_egp,transmission,color,city,seller_type,fuel_type
0,7284417,Toyota,Yaris,2009,170000.0,540000,Automatic,Dark blue,Heliopolis,dealer,Petrol
1,7296785,Toyota,Camry,2013,113000.0,1000000,Automatic,White,Nasr city,private,Petrol
2,7293331,Toyota,Corolla,2024,95000.0,1380000,Automatic,Black,Mohandessin,private,Petrol
3,7288941,Toyota,Corolla,2008,210000.0,585000,Automatic,Black,Heliopolis,dealer,Petrol
4,7295928,Toyota,Corolla,2022,126000.0,980000,Automatic,Dark red,Maadi,dealer,Petrol
...,...,...,...,...,...,...,...,...,...,...,...
3260,7243761,Hyundai,Excel,1997,470000.0,85000,Manual,Red,Shubra al-Khaimah,private,Petrol
3261,7243621,Hyundai,Verna,2007,166000.0,330000,Manual,Dark red,Damanhur,private,Petrol
3262,7243617,Hyundai,Getz,2008,162000.0,470000,Automatic,Red,Talkha,private,Petrol
3263,7243499,Hyundai,Getz,2011,200000.0,425000,Automatic,Dark red,Mansoura,private,Petrol


In [ ]:
X_train.to_csv('X_train_final.csv', index=False)
X_test.to_csv('X_test_final.csv', index=False)

y_train.to_csv('y_train_final.csv', index=False)
y_test.to_csv('y_test_final.csv', index=False)